In [0]:

-- ================================================================
-- EDA — USER PROFILES TABLE
-- ================================================================

-- See the whole table
SELECT *
FROM retail.default.user_profiles_bright_tv_dataset
LIMIT 10;

-- How many rows / how many unique subscribers
SELECT
    COUNT(*) AS number_of_rows,
    COUNT(DISTINCT UserID) AS number_of_subs
FROM retail.default.user_profiles_bright_tv_dataset;

-- Any duplicate UserIDs
SELECT UserID, COUNT(*) AS duplicate_count
FROM retail.default.user_profiles_bright_tv_dataset
GROUP BY UserID
HAVING COUNT(*) > 1;

-- Any NULL UserIDs
SELECT COUNT(*) AS cnt
FROM retail.default.user_profiles_bright_tv_dataset
WHERE UserID IS NULL;

-- Gender breakdown
SELECT Gender, COUNT(*) AS cnt
FROM retail.default.user_profiles_bright_tv_dataset
GROUP BY Gender
ORDER BY cnt DESC;

-- Race breakdown
SELECT Race, COUNT(*) AS cnt
FROM retail.default.user_profiles_bright_tv_dataset
GROUP BY Race
ORDER BY cnt DESC;

-- Province breakdown
SELECT Province, COUNT(*) AS cnt
FROM retail.default.user_profiles_bright_tv_dataset
GROUP BY Province
ORDER BY cnt DESC;

-- Age range
SELECT
    MIN(Age) AS min_age,
    MAX(Age) AS max_age
FROM retail.default.user_profiles_bright_tv_dataset;

-- How many subs have Age = 0 (likely missing/unknown, not real age)
SELECT COUNT(*) AS age_zero_count
FROM retail.default.user_profiles_bright_tv_dataset
WHERE Age = 0;

-- Any NULL ages
SELECT COUNT(*) AS cnt
FROM retail.default.user_profiles_bright_tv_dataset
WHERE Age IS NULL;

-- Email completeness
SELECT COUNT(*) AS missing_email
FROM retail.default.user_profiles_bright_tv_dataset
WHERE Email IS NULL OR TRIM(Email) = '' OR Email = 'None';

-- Social Media Handle completeness
SELECT COUNT(*) AS missing_social_handle
FROM retail.default.user_profiles_bright_tv_dataset
WHERE `Social Media Handle` IS NULL OR TRIM(`Social Media Handle`) = '' OR `Social Media Handle` = 'None';

-------------------------------------------------------------------------------

WITH Clean_Usership AS (

    SELECT
        UserID,
        Name,
        Surname,
        Email,

        CASE
            WHEN Gender = 'None' THEN 'Unknown'
            WHEN TRIM(Gender) = '' THEN 'Unknown'
            ELSE Gender
        END AS Gender,

        CASE
            WHEN Race = 'None' THEN 'Unknown'
            WHEN Race = 'other' THEN 'Unknown'
            WHEN TRIM(Race) = '' THEN 'Unknown'
            ELSE Race
        END AS Race,

        CASE
            WHEN Age = 0 THEN 'Unknown'
            WHEN Age BETWEEN 1 AND 12 THEN 'Kids'
            WHEN Age BETWEEN 13 AND 19 THEN 'Teenager'
            WHEN Age BETWEEN 20 AND 35 THEN 'Youth'
            WHEN Age BETWEEN 36 AND 50 THEN 'Adult'
            WHEN Age BETWEEN 51 AND 65 THEN 'Elder'
            ELSE 'Pensioner'
        END AS Age_Group,

        CASE
            WHEN Province = 'None' THEN 'Uncategorized'
            WHEN TRIM(Province) = '' THEN 'Uncategorized'
            ELSE Province
        END AS Province

    FROM retail.default.user_profiles_bright_tv_dataset

),

Clean_Viewership AS (

    SELECT
        UserID0 AS UserID,

        CASE
            WHEN LOWER(Channel2) = 'sawsee' THEN 'SawSee'
            WHEN Channel2 IN ('Supersport Live Events','SuperSport Live Events','Live on SuperSport') THEN 'SuperSport Live Events'
            ELSE Channel2
        END AS Tv_Channel,

        RecordDate2 + INTERVAL 2 HOURS AS Watch_Date_SA,
        DATE_FORMAT(RecordDate2 + INTERVAL 2 HOURS, 'E') AS Day_Name,
        HOUR(RecordDate2 + INTERVAL 2 HOURS) AS Hour_Of_Day,

        CASE
            WHEN DATE_FORMAT(RecordDate2 + INTERVAL 2 HOURS, 'E') IN ('Sat','Sun') THEN 'Weekend'
            ELSE 'Weekday'
        END AS Day_Classification,

        (HOUR(`Duration 2`) * 60 + MINUTE(`Duration 2`) + SECOND(`Duration 2`) / 60.0) AS Duration_Minutes,

        CASE
            WHEN Channel2 LIKE '%SuperSport%' OR Channel2 IN ('ICC Cricket World Cup 2011','Wimbledon') THEN 'Sports'
            WHEN Channel2 IN ('Cartoon Network','Boomerang') THEN 'Kids'
            WHEN Channel2 = 'CNN' THEN 'News'
            ELSE 'Entertainment'
        END AS Channel_Category

    FROM retail.default.viewership_bright_tv_dataset

)

SELECT
    u.UserID,
    u.Gender,
    u.Race,
    u.Age_Group,
    u.Province,
    v.Tv_Channel,
    v.Channel_Category,
    v.Watch_Date_SA,
    v.Day_Name,
    v.Day_Classification,
    v.Hour_Of_Day,
    v.Duration_Minutes

FROM Clean_Usership u
LEFT JOIN Clean_Viewership v
    ON u.UserID = v.UserID;